# Plate solving

## 필요한 환경

이 프로젝트를 위해서는 아래의 모듈이 필요합니다.

> numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, ysfitsutilpy, ysphotutilpy, version_information

### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, ysfitsutilpy, ysphotutilpy, version_information" # required modules
pkgs = packages.split(", ")

for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed... now start install")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"****** module {pkg} is installed")
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module ysfitsutilpy is installed
**** module ysphotutilpy is installed
**** module version_information is installed
This notebook was generated at 2024-07-08 20:47:50 (KST = GMT+0900) 
The Gaia ESA Archive will not be available due to an ESA-IT network intervention between 8th July 17:00 and 9th July 09:00 CEST.
0 Python     3.12.3 64bit [GCC 11.2.0]
1 IPython    8.25.0
2 OS         Linux 5.15.0 107 generic x86_64 with glibc2.31
3 numpy      1.26.4
4 pandas     2.2.2
5 matplotlib 3.8.4
6 scipy      1.13.0
7 astropy    6.1.0
8 photutils  1.12.0
9 ccdproc    2.4.2
10 ysfitsutilpy 0.2
11 ysphotutilpy 0.1.1
12 version_information 1.0.4


### import modules

In [2]:
from glob import glob
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData

from astroquery.astrometry_net import AstrometryNet

import ysfitsutilpy as yfu

import _astro_utilities
import _Python_utilities

# 환경 설정

In [3]:
#%%
#######################################################
BASEDIR = Path("/mnt/Rdata/OBS_data") 
PROJECDIR = Path("/mnt/Rdata/OBS_data/2024-EXO")
TODODIR = PROJECDIR / "_-_-_2024-05_-_GSON300_STF-8300M_-_1bin"
TODODIR = PROJECDIR / "_-_-_2024-06_-_GSON300_STF-8300M_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_2bin"

# PROJECDIR = Path("/mnt/Rdata/OBS_data/2022-Asteroid")
# TODODIR = PROJECDIR / "GSON300_STF-8300M_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_2bin"

# PROJECDIR = Path("/mnt/Rdata/OBS_data/2023-Asteroid")
# TODODIR = PROJECDIR / "GSON300_STF-8300M_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_2bin"

# PROJECDIR = Path("/mnt/Rdata/OBS_data/2016-Variable")
# TODODIR = PROJECDIR / "-_-_-_2016-_-_RiLA600_STX-16803_-_2bin"

# PROJECDIR = Path("/mnt/Rdata/OBS_data/2017-Variable")
# TODODIR = PROJECDIR / "-_-_-_2017-_-_RiLA600_STX-16803_-_2bin"

DOINGDIRs = sorted(_Python_utilities.getFullnameListOfsubDirs(TODODIR))
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

CALDIR = [x for x in DOINGDIRs if "CAL-BDF" in str(x)]
MASTERDIR = Path(CALDIR[0]) / _astro_utilities.master_dir
if not MASTERDIR.exists():
    os.makedirs("{}".format(str(MASTERDIR)))
    print("{} is created...".format(str(MASTERDIR)))

print ("MASTERDIR: ", format(MASTERDIR))

DOINGDIRs = sorted([x for x in DOINGDIRs if "_LIGHT_" in str(x)])
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

# filter_str = 'HAT-P-37b_LIGHT_-_2024-06-28_-_RiLA600_STX-16803_-_2bin'
# DOINGDIRs = [x for x in DOINGDIRs if filter_str in str(x)]
# remove = 'BIAS'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
# remove = 'DARK'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
# remove = 'FLAT'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
print ("DOINGDIRs: ", DOINGDIRs)
print ("len(DOINGDIRs): ", len(DOINGDIRs))
#######################################################

DOINGDIRs:  ['/mnt/Rdata/OBS_data/2024-EXO/_-_-_2024-06_-_GSON300_STF-8300M_-_1bin/-_CAL-BDF_-_2024-05_-_GSON300_STF-8300M_-_1bin/', '/mnt/Rdata/OBS_data/2024-EXO/_-_-_2024-06_-_GSON300_STF-8300M_-_1bin/HAT-P-37b_LIGHT_-_2024-06-28_-_GSON300_STF-8300M_-_1bin/', '/mnt/Rdata/OBS_data/2024-EXO/_-_-_2024-06_-_GSON300_STF-8300M_-_1bin/HD189733b_LIGHT_-_2024-06-16_-_GSON300_STF-8300M_-_1bin/', '/mnt/Rdata/OBS_data/2024-EXO/_-_-_2024-06_-_GSON300_STF-8300M_-_1bin/HD189733b_LIGHT_-_2024-06-27_-_GSON300_STF-8300M_-_1bin/', '/mnt/Rdata/OBS_data/2024-EXO/_-_-_2024-06_-_GSON300_STF-8300M_-_1bin/Kepler-17b_LIGHT_-_2024-06-26_-_GSON300_STF-8300M_-_1bin/', '/mnt/Rdata/OBS_data/2024-EXO/_-_-_2024-06_-_GSON300_STF-8300M_-_1bin/Qatar-10b_LIGHT_-_2024-06-02_-_GSON300_STF-8300M_-_1bin/', '/mnt/Rdata/OBS_data/2024-EXO/_-_-_2024-06_-_GSON300_STF-8300M_-_1bin/TrES-1b_LIGHT_-_2024-06-09_-_GSON300_STF-8300M_-_1bin/', '/mnt/Rdata/OBS_data/2024-EXO/_-_-_2024-06_-_GSON300_STF-8300M_-_1bin/TrES-1b_LIGHT_-_2024-06-

In [4]:
DOINGDIR = Path('/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin')
print("DOINGDIR", DOINGDIR)
SOLVINGDIR = DOINGDIR / _astro_utilities.reduced_dir
SOLVINGDIR = DOINGDIR / _astro_utilities.reduced_nightsky_dir
SOLVINGDIR = DOINGDIR

summary = yfu.make_summary(DOINGDIR/"*.fit*")
if summary is not None :
    print("len(summary):", len(summary))
    print("summary:", summary)
    #print(summary["file"][0])  
    df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
    df_light = df_light.reset_index(drop=True)
    print("df_light:\n{}".format(df_light))
df_light


DOINGDIR /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin
All 49 keywords (guessed from /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/CY-AQR_LIGHT_V_2016-09-22-12-59-31_30sec_RiLA600_STX-16803_-20c_2bin.fit) will be loaded.


DELMAG   =   0.0459            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =   0.0637            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =   0.0723            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =   0.0749            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =   0.0536            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =   0.0530            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =   0.0582            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =   0.0883            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:305: UserWarning: Key OBSERVAT not found for /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS

len(summary): 16
summary:                                                  file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...   8395200    True   
1   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...   8395200    True   
2   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...   8395200    True   
3   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...   8395200    True   
4   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...   8395200    True   
5   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...   8395200    True   
6   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...   8395200    True   
7   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...   8395200    True   
8   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...  16983360    True   
9   /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...  16983360    True   
10  /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...  16983360    True   
11  /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...  16983360    True   

/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:305: UserWarning: Key OBSERVAT not found for /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/M13_LIGHT_L_2022-04-02-20-14-44_600sec_FS60CB_STF-8300M_-30c_1bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:305: UserWarning: Key DETECTOR not found for /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/M13_LIGHT_L_2022-04-02-20-14-44_600sec_FS60CB_STF-8300M_-30c_1bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:305: UserWarning: Key TIME-OBS not found for /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/M13_LIGHT_L_2022-04-02-20-14-44_600sec_FS60CB_STF-8300M_-30c_1bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Down

,file,filesize,SIMPLE,BITPIX,NAXIS,NAXIS1,NAXIS2,BSCALE,BZERO,OBSERVAT,...,TELESCOP,XBINNING,YBINNING,EXPOSURE,TELESCOPE,APATURE,FLIPSTAT,XPIXSZ,YPIXSZ,PIXSCALE
0,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,8395200,True,16,2,2048,2048,1,32768,GSHS,...,-,2,2,30.0,-,600.0,,18.0,18.0,1.237590
1,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,8395200,True,16,2,2048,2048,1,32768,GSHS,...,-,2,2,30.0,-,600.0,,18.0,18.0,1.237590
2,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,8395200,True,16,2,2048,2048,1,32768,GSHS,...,-,2,2,30.0,-,600.0,,18.0,18.0,1.237590
3,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,8395200,True,16,2,2048,2048,1,32768,GSHS,...,-,2,2,30.0,-,600.0,,18.0,18.0,1.237590
4,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,8395200,True,16,2,2048,2048,1,32768,GSHS,...,-,2,2,30.0,-,600.0,,18.0,18.0,1.237590
5,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,8395200,True,16,2,2048,2048,1,32768,GSHS,...,-,2,2,30.0,-,600.0,,18.0,18.0,1.237590
6,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,8395200,True,16,2,2048,2048,1,32768,GSHS,...,-,2,2,30.0,-,600.0,,18.0,18.0,1.237590
7,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,8395200,True,16,2,2048,2048,1,32768,GSHS,...,-,2,2,30.0,-,600.0,,18.0,18.0,1.237590
8,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,16983360,True,16,2,3352,2532,1,32768,None,...,EQMOD ASCOM HEQ5/6,1,1,600.0,None,NaN,,5.4,5.4,3.137552
9,/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2...,16983360,True,16,2,3352,2532,1,32768,None,...,EQMOD ASCOM HEQ5/6,1,1,600.0,None,NaN,,5.4,5.4,3.137552


In [5]:
for _, row  in df_light.iterrows():

    fpath = Path(row["file"])
    # fpath = Path(df_light["file"][1])
    print("fpath :" ,fpath)
    hdul = fits.open(fpath)

    if 'PIXSCALE' in hdul[0].header:
        PIXc = hdul[0].header['PIXSCALE']
    else : 
        PIXc = _astro_utilities.calPixScale(hdul[0].header['FOCALLEN'], 
                                            hdul[0].header['XPIXSZ'],
                                            hdul[0].header['XBINNING'])
    print("PIXc : ", PIXc)
    hdul.close()

    solved = _astro_utilities.KevinSolver(fpath, 
                                            #str(SOLVEDDIR), 
                                            # downsample = 2,
                                            # pixscale = PIXc,
                                            tryASTAP = True, 
                                            tryLOCAL = True,
                                            tryASTROMETRYNET = False, 
                                            )


DELMAG   =   0.0459            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]


fpath : /mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/CY-AQR_LIGHT_V_2016-09-22-12-59-31_30sec_RiLA600_STX-16803_-20c_2bin.fit
PIXc :  1.23759
pixscale: 1.238, L: 1.200, U: 1.275
SOLVE: False ASTAP: False LOCAL: False
Trying to solve using ASTAP: CY-AQR_LIGHT_V_2016-09-22-12-59-31_30sec_RiLA600_STX-16803_-20c_2bin.fit 
b'[WARNING] GetPosition called without handle for PairSplitter1(TPairSplitter)\nUsing star database D50\nCreating grayscale x 2 binning image for solving or star alignment.\n13 stars, 9 quads selected in the image. 13 database stars, 9 database quads required for the square search field of 0.7\xc2\xb0. Search window at 200% based on the number of quads. Step size at 100% of image height.\nNo solution found!  :(\n'
SOLVE: False ASTAP: False LOCAL: False
Trying to solve using LOCAL: CY-AQR_LIGHT_V_2016-09-22-12-59-31_30sec_RiLA600_STX-16803_-20c_2bin.fit 


DELMAG   =   0.0459            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]


b'Reading input file 1 of 1: "/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/CY-AQR_LIGHT_V_2016-09-22-12-59-31_30sec_RiLA600_STX-16803_-20c_2bin.fit"...\nExtracting sources...\nDownsampling by 2...\nsimplexy: found 402 sources.\nSolving...\nReading file "/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/CY-AQR_LIGHT_V_2016-09-22-12-59-31_30sec_RiLA600_STX-16803_-20c_2bin.axy"...\nField 1 did not solve (index index-4219.fits, field objects 1-10).\nField 1 did not solve (index index-4218.fits, field objects 1-10).\nField 1 did not solve (index index-4217.fits, field objects 1-10).\nField 1 did not solve (index index-4216.fits, field objects 1-10).\nField 1 did not solve (index index-4215.fits, field objects 1-10).\nField 1 did not solve (index index-4214.fits, field objects 1-10).\nField 1 did not solve (index index-4213.fits, field objects 1-10).\nField 1 did not solve (index index-4212.fits, field objects 1-10).\nField

DELMAG   =   0.0637            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]


...

b'Reading input file 1 of 1: "/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/CY-AQR_LIGHT_V_2016-09-22-13-03-55_30sec_RiLA600_STX-16803_-20c_2bin.fit"...\nExtracting sources...\nDownsampling by 2...\nsimplexy: found 402 sources.\nSolving...\nReading file "/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/CY-AQR_LIGHT_V_2016-09-22-13-03-55_30sec_RiLA600_STX-16803_-20c_2bin.axy"...\nField 1 did not solve (index index-4219.fits, field objects 1-10).\nField 1 did not solve (index index-4218.fits, field objects 1-10).\nField 1 did not solve (index index-4217.fits, field objects 1-10).\nField 1 did not solve (index index-4216.fits, field objects 1-10).\nField 1 did not solve (index index-4215.fits, field objects 1-10).\nField 1 did not solve (index index-4214.fits, field objects 1-10).\nField 1 did not solve (index index-4213.fits, field objects 1-10).\nField 1 did not solve (index index-4212.fits, field objects 1-10).\nField

DELMAG   =   0.0883            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]


b'[WARNING] GetPosition called without handle for PairSplitter1(TPairSplitter)\nUsing star database D50\nCreating grayscale x 2 binning image for solving or star alignment.\n12 stars, 7 quads selected in the image. 12 database stars, 7 database quads required for the square search field of 0.7\xc2\xb0. Search window at 200% based on the number of quads. Step size at 100% of image height.\nNo solution found!  :(\n'
SOLVE: False ASTAP: False LOCAL: False
Trying to solve using LOCAL: CY-AQR_LIGHT_V_2016-09-22-13-04-39_30sec_RiLA600_STX-16803_-20c_2bin.fit 


DELMAG   =   0.0883            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]


b'Reading input file 1 of 1: "/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/CY-AQR_LIGHT_V_2016-09-22-13-04-39_30sec_RiLA600_STX-16803_-20c_2bin.fit"...\nExtracting sources...\nDownsampling by 2...\nsimplexy: found 380 sources.\nSolving...\nReading file "/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin/CY-AQR_LIGHT_V_2016-09-22-13-04-39_30sec_RiLA600_STX-16803_-20c_2bin.axy"...\nField 1 did not solve (index index-4219.fits, field objects 1-10).\nField 1 did not solve (index index-4218.fits, field objects 1-10).\nField 1 did not solve (index index-4217.fits, field objects 1-10).\nField 1 did not solve (index index-4216.fits, field objects 1-10).\nField 1 did not solve (index index-4215.fits, field objects 1-10).\nField 1 did not solve (index index-4214.fits, field objects 1-10).\nField 1 did not solve (index index-4213.fits, field objects 1-10).\nField 1 did not solve (index index-4212.fits, field objects 1-10).\nField

In [6]:
str(fpath.parents[0])
fpath.parent

PosixPath('/mnt/Rdata/OBS_data/CCD_obs_test/M13_LIGHT_-_2022-04-02_-_FS60CB_STF-8300M_-_1bin')

In [7]:
# for _, row  in df_light.iterrows():

#     fpath = Path(row["file"])
#     # os.makedirs(fpath.parents[0]/"wcs_remove")
#     yfu.wcsremove(fpath, 
#                 additional_keys=["COMMENT", "HISTORY"],
#                 verbose=True,
#                 output=fpath.parents[0]/"wcs_remove"/fpath.name,
#                 ccddata=False,
#                 overwrite=True)

In [8]:
# fpath = Path(df_light["file"][2])
# if not SOLVE :
#     print(f"{fpath.name} is solving now by ASTAP")
#     solved = _astro_utilities.KevinSolver(fpath, 
#                                             #str(SOLVEDDIR), 
#                                             # downsample = 2,
#                                             pixscale = PIXc,
#                                             tryASTAP = True, 
#                                             tryLOCAL = True,
#                                             tryASTROMETRYNET = False, 
#                                             )

In [9]:
# fpath = Path(df_light["file"][2])
# if not SOLVE :
#     print(f"{fpath.name} is solving now by ASTAP")
#     solved = _astro_utilities.ASTAPSolver(fpath, 
#                                             #str(SOLVEDDIR), 
#                                             # downsample = 2,
#                                             pixscale = PIXc,
#                                             )

In [10]:
# for DOINGDIR in DOINGDIRs[:1] :
#     DOINGDIR = Path(DOINGDIR)
#     print("DOINGDIR", DOINGDIR)
#     if str(DOINGDIR.parts[-2]) == "RiLA600_STX-16803_-_1bin" :
#         DOINGDIR = DOINGDIR / _astro_utilities.reduced_nightsky_dir
#     if str(DOINGDIR.parts[-2]) == "GSON300_STF-8300M_-_1bin" :
#         DOINGDIR = DOINGDIR / _astro_utilities.reduced_dir
    
#     fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
#     #print("fits_in_dir", fits_in_dir)
#     print("len(fits_in_dir)", len(fits_in_dir))

#     if len(fits_in_dir) == 0 :
#         print(f"There is no fits fils in {DOINGDIR}")
#         pass
#     else : 
#         print(f"Starting: {str(DOINGDIR.parts[-1])}")

#         summary = yfu.make_summary(DOINGDIR/"*.fit*")
#         print("len(summary):", len(summary))
#         print("summary:", summary)
#         #print(summary["file"][0])
#         df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
#         df_light = df_light.reset_index(drop=True)
#         print("df_light:\n{}".format(df_light))

#         for _, row  in df_light.iterrows():

#             fpath = Path(row["file"])
#             hdul = fits.open(fpath)
            
#             if 'PIXSCALE' in hdul[0].header:
#                 PIXc = hdul[0].header['PIXSCALE']
#             else : 
#                 PIXc = _astro_utilities.calPixScale(hdul[0].header['FOCALLEN'], 
#                                                     hdul[0].header['XPIXSZ'],
#                                                     hdul[0].header['XBINNING'])
#             print("PIXc : ", PIXc)
#             hdul.close()

#             SOLVE, ASTAP, LOCAL = _astro_utilities.checkPSolve(fpath)
#             print("SOLVE:", SOLVE, "ASTAP:", ASTAP, "LOCAL:", LOCAL)

#             if not SOLVE :
#                 print(f"{fpath.name} is solving now by ASTAP")
#                 solved = _astro_utilities.ASTAPSolver(fpath, 
#                                                         #str(SOLVEDDIR), 
#                                                         downsample = 2,
#                                                         pixscale = PIXc,
#                                                                 )

#                 SOLVE, ASTAP, LOCAL = _astro_utilities.checkPSolve(fpath)
#                 print("SOLVE:", SOLVE, "ASTAP:", ASTAP, "LOCAL:", LOCAL)

#                 if not SOLVE : 
#                     print(f"{fpath.name} is solving now by LOCAL")
#                     if 'PIXSCALE' in hdul[0].header:
#                         PIXc = hdul[0].header['PIXSCALE']
#                     else : 
#                         PIXc = _astro_utilities.calPixScale(hdul[0].header['FOCALLEN'], 
#                                                         hdul[0].header['XPIXSZ'],
#                                                         hdul[0].header['XBINNING'])
#                     print("PIXc : ", PIXc)

#                     solved = _astro_utilities.LOCALPSolver(fpath, 
#                                                             #str(SOLVEDDIR), 
#                                                             downsample = 2,
#                                                             pixscale = PIXc,
#                                                                     )